In [16]:

import pandas as pd
#get the data
path = "../data/processed/MBP10_ZC.parquet"
df = pd.read_parquet(path, columns=["ts_event", "bid_px_05", "ask_px_05"])
df = df.sort_values("ts_event").reset_index(drop=True)

df 



,ts_event,bid_px_05,ask_px_05
0,2025-02-16 13:00:06.432206309+00:00,495.5,497.00
1,2025-02-16 22:00:00.337137947+00:00,495.5,497.00
2,2025-02-16 22:00:22.296140633+00:00,495.5,497.00
3,2025-02-16 22:00:22.538433281+00:00,495.5,497.00
4,2025-02-16 22:00:23.483635199+00:00,495.5,497.00
...,...,...,...
30839149,2026-02-15 22:00:10.648420233+00:00,431.0,431.75
30839150,2026-02-15 22:00:11.228268619+00:00,431.0,431.75
30839151,2026-02-15 22:00:12.104053681+00:00,431.0,431.75
30839152,2026-02-15 22:00:12.716769795+00:00,431.0,431.50


In [2]:
import pandas as pd
import plotly.graph_objects as go

MBP10_data_path = "../data/processed/MBP10_ZC.parquet"


def create_bid_ask_graph(bid_order_book_level: int = 0,ask_order_book_level: int = 0):

    if max(bid_order_book_level,ask_order_book_level) > 5 or min(ask_order_book_level,bid_order_book_level) < 0: #just some input validation
        return "choose and orderbook level between 0 and 5 inclusive"
    
    bid_column = "bid_px_0" + str(bid_order_book_level)
    ask_column = "ask_px_0" + str(ask_order_book_level)

    df = pd.read_parquet(MBP10_data_path, columns = ["ts_event", bid_column, ask_column]).sort_values("ts_event").reset_index(drop=True)
    df_plot = (
    df.set_index("ts_event")
    .resample("5min")[[bid_column, ask_column]]
    .last()
    .dropna()
    .reset_index())

    # plot
    y_pad = (df_plot[ask_column].max() - df_plot[bid_column].min()) * 0.05

    fig = go.Figure()

    # Ask first — so fill direction goes ask -> bid
    fig.add_trace(go.Scatter(
        x=df_plot["ts_event"], y=df_plot[ask_column],
        name="Ask lvl " + str(ask_order_book_level),
        line=dict(color="#EF5350", width=1.8),
        mode="lines"
    ))

    # Bid 
    fig.add_trace(go.Scatter(
        x=df_plot["ts_event"], y=df_plot[bid_column],
        name="Bid lvl " + str(bid_order_book_level),
        line=dict(color="#26A65B", width=1.8),
        fill="tonexty",
        fillcolor="rgba(160, 160, 160, 0.15)",
        mode="lines"
    ))

    fig.update_layout(
        title=dict(
            text="Bid/Ask Price (Level" + str(bid_order_book_level)+ "/" + str(ask_order_book_level)+  ") — ZC MBP10<br><span style='font-size:15px;font-weight:normal;'>1-Year | 5-min resampled | Spread shaded</span>",
            x=0.5, xanchor="center"
        ),
        legend=dict(orientation="h", yanchor="top", y=-0.12, xanchor="center", x=0.5),
        hovermode="x unified",
        yaxis=dict(
            range=[df_plot[bid_column].min() - y_pad, df_plot[ask_column].max() + y_pad],
            showgrid=True, gridcolor="rgba(128,128,128,0.12)", zeroline=False,
        ),
        xaxis=dict(
            showgrid=True, gridcolor="rgba(128,128,128,0.12)",
            dtick="M1", tickformat="%b '%y",
        ),
        margin=dict(l=70, r=40, t=110, b=80),
    )

    fig.update_yaxes(title_text="Price (USD/bu)")
    fig.show()

        

In [10]:
# bid_0,ask_0 = 0,0
# create_bid_ask_graph(bid_0,ask_0)

### Bid-Ask level 0: ZC 1y
![Image of level 5 bid ask of corn](../outputs/charts/bidask0.png)

In [11]:
# bid_1,ask_1 = 1,1
# create_bid_ask_graph(bid_1,ask_1)

### Bid-Ask level 1: ZC 1y
![Image of level 5 bid ask of corn](../outputs/charts/bidask1.png)


In [12]:
# bid_2,ask_2 = 2,2
# create_bid_ask_graph(bid_2,ask_2)

### Bid-Ask level 2: ZC 1y
![Image of level 5 bid ask of corn](../outputs/charts/bidask2.png)


In [13]:
# bid_3,ask_3 = 3,3
# create_bid_ask_graph(bid_3,ask_3)

### Bid-Ask level 3: ZC 1y
![Image of level 5 bid ask of corn](../outputs/charts/bidask3.png)


In [14]:
# bid_4,ask_4 = 4,4
# create_bid_ask_graph(bid_4,ask_4)

### Bid-Ask level 4: ZC 1y
![Image of level 5 bid ask of corn](../outputs/charts/bidask4.png)


In [15]:
# bid_5,ask_5 = 5,5
# create_bid_ask_graph(bid_5,ask_5)

### Bid-Ask level 5: ZC 1y
![Image of level 5 bid ask of corn](../outputs/charts/bidask5.png)


## Intraday Analysis
- looking at the daily file with the most action (based on rows in the dataset - 366,473): 2026-01-12

In [27]:
daily_counts = df.groupby(df['ts_event'].dt.date).size()
most_active_day = daily_counts.idxmax()
print(f"most active day: {most_active_day} with {daily_counts.max():,} rows")


most active day: 2026-01-12 with 366,473 rows


In [30]:
data_20260112 = df[df['ts_event'].dt.date == pd.Timestamp('2026-01-12').date()]

In [31]:
data_20260112

,ts_event,bid_px_05,ask_px_05
26674982,2026-01-12 00:15:22.944863117+00:00,445.50,444.75
26674983,2026-01-12 00:15:24.459555255+00:00,445.50,444.75
26674984,2026-01-12 00:27:29.501444685+00:00,445.50,444.75
26674985,2026-01-12 00:35:41.992694173+00:00,445.75,444.75
26674986,2026-01-12 00:45:00.024087465+00:00,445.75,444.75
...,...,...,...
27041450,2026-01-12 23:12:54.144329447+00:00,422.25,420.00
27041451,2026-01-12 23:27:44.695182433+00:00,422.50,420.00
27041452,2026-01-12 23:42:22.783438217+00:00,423.00,420.00
27041453,2026-01-12 23:42:25.866061207+00:00,422.50,420.00


In [89]:
import plotly.graph_objects as go


def plot_intraday_volume(df: pd.DataFrame, date: str = "20260112", bar_size: str = "5min") -> go.Figure | str:
    """
    plot intraday tick volume as a bar chart for a given date.
    """
    df = df.copy()
    df["ts_event"] = pd.to_datetime(df["ts_event"], utc=True)

    #input validation
    try:
        target_date = pd.Timestamp(date, tz="UTC").date()
    except ValueError:
        return f" date format wrong '{date}'. should be YYYYMMDD (e.g. '20250304')."

    one_day = df[df["ts_event"].dt.date == target_date]

    if one_day.empty:
        return f"No data found for date {target_date} ->check that this date exists in the dataset."

    volume = one_day.set_index("ts_event").resample(bar_size).size()
    volume = volume[volume > 0]  # drop empty buckets outside trading hours

    fig = go.Figure(
        go.Bar(
            x=volume.index,
            y=volume.values,
            name="Tick Count",
            marker_color="#35664d",
        )
    )
    fig.add_vline(
        x='2026-01-12 14:30:00.459555255',
        #annotation_text="market open",
        line_dash="dot",
        line_color="#CAC862",
        line_width=2,
    )
    fig.add_vline(
        x='2026-01-12 19:20:00.459555255',
        #annotation_text="market close",
        line_dash="dot",
        line_color="#CAC862",
        line_width=2,
    )
    fig.add_vline(
        x='2026-01-12 01:00:00.459555255',
        #annotation_text="globex open",
        line_dash="dot",
        line_color="#CAC862",
        line_width=2,
    )
    fig.add_vline(
        x='2026-01-12 13:45:00.459555255',
        #annotation_text="overnight close",
        line_dash="dot",
        line_color="#CAC862",
        line_width=2,
    )
    fig.add_vline(
        x='2026-01-12 19:15:00.459555255',
        #annotation_text="settlement",
        line_dash="dot",
        line_color="#CAC862",
        line_width=2,
    )

    fig.update_layout(
        title=f"Intraday Volume — {target_date} ({bar_size} buckets)",
        xaxis_title="Time",
        yaxis_title="Tick Count",
        xaxis=dict(type="date"),
        bargap=0.1,
    )
    """ 
    Globex Open:     2026-01-12 01:00:00
    Overnight Close: 2026-01-12 13:45:00
    Day Open:        2026-01-12 14:30:00
    Settlement:      2026-01-12 19:15:00
    Day Close:       2026-01-12 19:20:00

    Significant times like market open, close, settlement, etc are 1 hour earlier after the first sunday of March.



    """
    return fig


In [91]:
#plot_intraday_volume(df, date="20260112", bar_size="3min")


### Intraday ZC volume (3min): 2026-01-12
![Image of level 5 bid ask of corn](../outputs/charts/intraday_volume_3m_20260112.png)